# Demo 04. Numerical Differentiation and the Optimal Step Size

**Module 2** (numerical differentiation), the step-size study of section 2.4.

A finite-difference derivative carries two competing errors. Truncation error comes from replacing the limit $h\to0$ with a fixed $h$ and shrinks as $h$ decreases. Rounding error comes from subtracting two nearly equal function values and *grows* as $h$ decreases. Their sum is a U-shaped curve with a finite optimal step size. This demo measures that curve against a high-precision reference and confirms the predicted location of its minimum.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
import mpmath as mp
from ipywidgets import interact, Dropdown, FloatSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)
mp.mp.dps = 50   # 50-digit reference for the exact derivative

## The two error terms

For the forward difference $D_+f(x)=\dfrac{f(x+h)-f(x)}{h}$, Taylor expansion gives a truncation error $\tfrac{h}{2}f''(\xi)$, while the stored function values each carry absolute rounding error near $u\lvert f\rvert$, where $u\approx1.1\times10^{-16}$ is the unit roundoff. The subtraction divides by $h$, so the rounding contribution is about $2u\lvert f\rvert/h$. The total error model is

$$ E_+(h) \approx \tfrac{h}{2}\lvert f''\rvert + \frac{2u\lvert f\rvert}{h}, \qquad h_{*} \approx 2\sqrt{\frac{u\lvert f\rvert}{\lvert f''\rvert}} = O(\sqrt u)\approx 10^{-8}. $$

The central difference $D_0 f(x)=\dfrac{f(x+h)-f(x-h)}{2h}$ has truncation error $\tfrac{h^2}{6}f'''$, so

$$ E_0(h) \approx \tfrac{h^2}{6}\lvert f'''\rvert + \frac{u\lvert f\rvert}{h}, \qquad h_{*} \approx \left(\frac{3u\lvert f\rvert}{\lvert f'''\rvert}\right)^{1/3} = O(u^{1/3})\approx 10^{-5}. $$

The central formula reaches a smaller minimum error at a larger step size.

In [ ]:
# ---------------------------------------------------------------------------
# The reference derivative, at 50 digits
# ---------------------------------------------------------------------------
# We differentiate in ordinary float64 but compare against a value computed by
# mpmath, so the measured error is the true error of the float computation and
# not an artifact of an approximate "exact" answer. Everything below
# differentiates exp at x = 1, where the answer is e.

mp.mp.dps = 50
X0 = 1.0
EXACT = float(mp.e)
U = np.finfo(float).eps            # unit roundoff, about 2.2e-16

print(f"f(x) = exp(x),  x0 = {X0},  f'(x0) = e = {EXACT:.15f}")
print(f"unit roundoff u = {U:.3e}")

In [ ]:
# ---------------------------------------------------------------------------
# The two difference quotients, swept over a list of step sizes
# ---------------------------------------------------------------------------
# Forward uses two samples one step apart; central straddles the point with a
# sample on each side. Written as a loop over the step sizes so each estimate
# says which h produced it.

def difference_sweep(x, hs, n):
    """Forward and central difference estimates of f'(x) at each step size."""
    fwd = np.zeros(n)
    cen = np.zeros(n)
    for k in range(n):
        h = hs[k]
        fwd[k] = (np.exp(x + h) - np.exp(x)) / h
        cen[k] = (np.exp(x + h) - np.exp(x - h)) / (2 * h)
    return fwd, cen

HS_DEMO = [0.2, 0.1, 0.05, 0.025]
f_est, c_est = difference_sweep(X0, HS_DEMO, 4)
print(f"{'h':>8} {'forward':>14} {'central':>16} {'central error':>16} {'ratio':>7}")
prev = None
for k, h in enumerate(HS_DEMO):
    e = abs(c_est[k] - EXACT)
    r = f"{prev / e:7.2f}" if prev else "      -"
    print(f"{h:8.3f} {f_est[k]:14.9f} {c_est[k]:16.11f} {e:16.3e} {r}")
    prev = e
print("\nHalving h divides the central error by 4, which is the h^2 in the theory.")

In [ ]:
# ---------------------------------------------------------------------------
# Sweep h over sixteen orders of magnitude: the U-curve
# ---------------------------------------------------------------------------
# Each branch is a straight line on log-log axes. The truncation branch has the
# slope of the method's order; the rounding branch has slope -1, because both
# formulas divide a fixed rounding floor by h.
#
# Below the minimum the curve is spiky rather than smooth: there, the error is
# the leftover of a cancellation and it swings by an order of magnitude between
# neighbouring h. So the *pointwise* minimum of a sampled curve is the minimum
# of that noise and is not reproducible. The envelope below takes a median per
# half-decade instead, which is what the theory actually predicts.

def sweep(estimator, lo=-16.0, hi=0.0, count=4000):
    hs = np.logspace(lo, hi, count)
    err = np.array([abs(estimator(h) - EXACT) for h in hs])
    return hs, err

fwd_of = lambda h: (np.exp(X0 + h) - np.exp(X0)) / h
cen_of = lambda h: (np.exp(X0 + h) - np.exp(X0 - h)) / (2 * h)

def envelope(hs, err, per_decade=2):
    """Median error in each half-decade, and where that median bottoms out."""
    edges = np.logspace(np.log10(hs[0]), np.log10(hs[-1]),
                        int(per_decade * np.log10(hs[-1] / hs[0])) + 1)
    mids, meds = [], []
    for a, b in zip(edges[:-1], edges[1:]):
        m = (hs >= a) & (hs < b)
        if m.sum() >= 5:
            mids.append(np.sqrt(a * b)); meds.append(np.median(err[m]))
    mids, meds = np.array(mids), np.array(meds)
    k = int(np.argmin(meds))
    return mids, meds, mids[k], meds[k]

h_f, e_f = sweep(fwd_of)
h_c, e_c = sweep(cen_of)
mf, vf, hf_star, ef_star = envelope(h_f, e_f)
mc, vc, hc_star, ec_star = envelope(h_c, e_c)

plt.figure()
plt.loglog(h_f, e_f + 1e-20, ".", ms=1.5, color="salmon", alpha=0.5)
plt.loglog(h_c, e_c + 1e-20, ".", ms=1.5, color="lightsteelblue", alpha=0.5)
plt.loglog(mf, vf, "r-", lw=2, label="forward  O(h)")
plt.loglog(mc, vc, "b-", lw=2, label="central  O(h^2)")
plt.axvline(np.sqrt(U), color="r", ls=":", lw=1)
plt.axvline(U ** (1 / 3), color="b", ls=":", lw=1)
plt.xlabel("step size h"); plt.ylabel("absolute error")
plt.title("Error against step size (dots: samples, lines: envelope)")
plt.legend(); plt.show()

print(f"forward: envelope bottoms at {ef_star:.2e} near h = {hf_star:.1e}"
      f"   (predicted h* ~ sqrt(u) = {np.sqrt(U):.1e})")
print(f"central: envelope bottoms at {ec_star:.2e} near h = {hc_star:.1e}"
      f"   (predicted h* ~ u^(1/3) = {U ** (1 / 3):.1e})")
print("\nShrinking h past the minimum makes the answer worse, not better.")

## Richardson extrapolation

The central difference has an error expansion in even powers of $h$,

$$D_0(h) = f'(x) + c_2h^2 + c_4h^4 + \cdots,$$

so evaluating at $h$ and at $h/2$ gives two estimates whose leading errors differ by a known factor of four. Any standard numerical analysis text carries the derivation; the useful consequence is that the combination

$$\frac{4D_0(h/2) - D_0(h)}{3}$$

cancels the $h^2$ term exactly and leaves an $O(h^4)$ approximation. That buys two extra orders for two extra function evaluations, and, more to the point here, it reaches its accuracy at a much larger $h$ than the central difference needs.

In [ ]:
# ---------------------------------------------------------------------------
# Richardson extrapolation of the central difference
# ---------------------------------------------------------------------------
def richardson_sweep(x, hs, n):
    """Extrapolated estimates of f'(x), one per step size."""
    out = np.zeros(n)
    for k in range(n):
        h = hs[k]
        coarse = (np.exp(x + h) - np.exp(x - h)) / (2 * h)
        fine = (np.exp(x + h / 2) - np.exp(x - h / 2)) / h
        out[k] = (4 * fine - coarse) / 3
    return out

r_est = richardson_sweep(X0, HS_DEMO, 4)
print(f"{'h':>8} {'Richardson':>18} {'error':>13} {'ratio':>7}")
prev = None
for k, h in enumerate(HS_DEMO):
    e = abs(r_est[k] - EXACT)
    r = f"{prev / e:7.2f}" if prev else "      -"
    print(f"{h:8.3f} {r_est[k]:18.13f} {e:13.3e} {r}")
    prev = e
print("\nHalving h now divides the error by 16, which is the h^4 in the theory.")

In [ ]:
# ---------------------------------------------------------------------------
# What extrapolation buys, and what it does not
# ---------------------------------------------------------------------------
# Richardson steepens the truncation branch from slope 2 to slope 4. It does
# not touch the rounding branch, which still rises like 1/h; combining two
# estimates in fact makes the rounding floor slightly worse. So the U does not
# go away. It moves: the minimum slides right, to a larger h, and down.

rich_of = lambda h: (4 * ((np.exp(X0 + h / 2) - np.exp(X0 - h / 2)) / h)
                     - (np.exp(X0 + h) - np.exp(X0 - h)) / (2 * h)) / 3
h_r, e_r = sweep(rich_of)
mr, vr, hr_star, er_star = envelope(h_r, e_r)

plt.figure()
for hh, ee, c in ((h_c, e_c, "lightsteelblue"), (h_r, e_r, "lightgreen")):
    plt.loglog(hh, ee + 1e-20, ".", ms=1.5, color=c, alpha=0.5)
plt.loglog(mc, vc, "b-", lw=2, label="central  O(h^2)")
plt.loglog(mr, vr, "g-", lw=2, label="Richardson  O(h^4)")
plt.xlabel("step size h"); plt.ylabel("absolute error")
plt.title("Richardson moves the minimum right and down")
plt.legend(); plt.show()

print(f"central   : best error {ec_star:.2e} at h = {hc_star:.1e}")
print(f"Richardson: best error {er_star:.2e} at h = {hr_star:.1e}")
print(f"\n{ec_star / er_star:.0f} times more accurate, "
      f"at a step {hr_star / hc_star:.0f} times larger.")
print("The rounding branch is unchanged, so the floor is lower but still there.")

for name, hh, ee in (("central", h_c, e_c), ("Richardson", h_r, e_r)):
    m = (hh > 1e-2) & (hh < 1e-1)
    print(f"fitted truncation slope, {name:11s}: "
          f"{np.polyfit(np.log(hh[m]), np.log(ee[m]), 1)[0]:.2f}")

In [ ]:
# ---------------------------------------------------------------------------
# Interactive: change the point and watch the picture hold
# ---------------------------------------------------------------------------
def show_at(x0=1.0):
    """The same comparison at a different point."""
    exact = float(mp.e ** mp.mpf(x0))
    cen = lambda h: (np.exp(x0 + h) - np.exp(x0 - h)) / (2 * h)
    rich = lambda h: (4 * ((np.exp(x0 + h / 2) - np.exp(x0 - h / 2)) / h)
                      - (np.exp(x0 + h) - np.exp(x0 - h)) / (2 * h)) / 3
    for name, g in (("central", cen), ("Richardson", rich)):
        hs = np.logspace(-16, 0, 3000)
        err = np.array([abs(g(h) - exact) for h in hs])
        _, _, hstar, estar = envelope(hs, err)
        print(f"  {name:11s}: best error {estar:.2e} at h = {hstar:.1e}")

interact(show_at, x0=FloatSlider(value=1.0, min=0.2, max=3.0, step=0.2,
                                 description="x0"));

## Summary

- A finite-difference derivative carries two errors that pull in opposite directions: truncation falls with $h$, rounding grows like $1/h$ because a fixed cancellation error is divided by a shrinking step. Their sum has a finite minimum.
- Forward differencing is $O(h)$ and bottoms out near $h_{*}\sim\sqrt{u}$; central differencing is $O(h^2)$ and bottoms out near $h_{*}\sim u^{1/3}$, at a smaller error.
- Below the minimum the error curve is not smooth. It is the leftover of a cancellation, swinging by an order of magnitude between neighbouring $h$, so a pointwise minimum of a sampled curve measures noise. The envelope is what the theory predicts.
- Richardson extrapolation combines estimates at $h$ and $h/2$ to cancel the leading error term, raising the order from $2$ to $4$ and dividing the error by $16$ at each halving instead of $4$.
- It steepens the truncation branch and leaves the rounding branch alone, so the U-curve does not disappear. Its minimum moves to a larger step and a smaller error, which is the useful direction.